# ⚾ LG Aimers 투수 제구 성공 확률 예측 — 55개 고정 피처 모델 튜닝 & 앙상블 (Kaggle / Colab / Local 호환)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seokjin-wq/Aimers-9th/blob/main/LG_Aimers_model_tuning_55features.ipynb)

---

## 📌 프로젝트 개요 및 환경 지원

본 노트북은 **Kaggle GPU, Google Colab GPU, 로컬 환경**에서 모두 원활하게 실행되도록 설계된 종합 모델 튜닝 및 DACON 제출 파일 생성 파이프라인입니다.

### 🛡️ 절대 준수 원칙 (Strict Constraints)
1. **피처 55개 완전 고정**: 검증된 `baseline` 55개 피처(3개 범주형 + 52개 수치형)를 일체 변경하지 않습니다.
2. **Kaggle / Colab 환경 자동 감지**:
   - `/kaggle/input` 및 `/kaggle/working` 자동 감지 및 재귀 탐색 지원
   - `open.zip` 발견 시 `/kaggle/working/lg_data`로 자동 압축 해제
   - 모든 산출물은 `artifacts/` 및 `reports/`에 자동 보관
3. **CatBoost GPU 가속 & `bootstrap_type="Bernoulli"` 명시**:
   - Kaggle / Colab GPU 환경에서 `task_type="GPU"`, `devices="0"` 사용
   - `subsample` 설정 시 `bootstrap_type="Bernoulli"`를 명시하여 GPU/CPU 호환성 보장
4. **결과 임의 생성 금지**: 사전 결과표 조작 없이, 실제 셀 실행을 통해 동적으로 검증 및 보고서를 생성합니다.
5. **네이티브 모델 & 1:1 매핑 제출**:
   - CatBoost(`.cbm`) 및 LightGBM(`.txt`) 네이티브 형식과 전처리 `metadata.json` 저장
   - `sample_submission.csv`의 `row_id`와 1:1 매핑되는 독립 추론 `script.py` 자동 생성 및 Smoke Test 검증


## 🛠️ 1. 환경 감지, 패키지 설치 및 디바이스 설정

Kaggle, Colab, 로컬 환경을 자동 판별하고 필요한 라이브러리를 설치한 뒤 GPU 가속을 활성화합니다.


In [ ]:
# 필수 머신러닝 라이브러리 설치 (Kaggle / Colab 호환)
!pip install -q catboost==1.2.10 lightgbm==4.7.0 xgboost==3.4.0 optuna


In [ ]:
import os
import sys
import gc
import time
import json
import shutil
import zipfile
import hashlib
import inspect
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import brier_score_loss, roc_auc_score

import catboost
from catboost import CatBoostClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
import xgboost as xgb
from xgboost import XGBClassifier

# 경고 무시 및 기본 시드 고정
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

# 환경 자동 감지 (Kaggle / Colab / Local)
if Path("/kaggle/working").exists():
    ENV_TYPE = "kaggle"
    BASE_WORKING_DIR = Path("/kaggle/working")
elif Path("/content").exists():
    ENV_TYPE = "colab"
    BASE_WORKING_DIR = Path("/content")
else:
    ENV_TYPE = "local"
    BASE_WORKING_DIR = Path(".")

ARTIFACT_DIR = BASE_WORKING_DIR / "artifacts"
REPORTS_DIR = BASE_WORKING_DIR / "reports"
BUILD_DIR = ARTIFACT_DIR / "submit_build"
MODEL_DIR = BUILD_DIR / "model"
ZIP_PATH = ARTIFACT_DIR / "submit_v02.zip"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print(f"📌 실행 환경: {ENV_TYPE.upper()} (Working Dir: {BASE_WORKING_DIR.resolve()})")
print(f"  • Python: {sys.version.split()[0]}")
print(f"  • pandas: {pd.__version__} | numpy: {np.__version__} | scikit-learn: {sklearn.__version__}")
print(f"  • CatBoost: {catboost.__version__} | LightGBM: {lgb.__version__} | XGBoost: {xgb.__version__}")
print(f"  • CPU Physical / Logical Cores: {psutil.cpu_count(logical=False)} / {psutil.cpu_count(logical=True)}")
ram = psutil.virtual_memory()
print(f"  • Total RAM: {ram.total / (1024**3):.2f} GB | Available: {ram.available / (1024**3):.2f} GB")

# GPU 사용 가능 여부 확인
def detect_gpu_environment():
    has_gpu = False
    gpu_name = "N/A"
    try:
        import torch
        if torch.cuda.is_available():
            has_gpu = True
            gpu_name = torch.cuda.get_device_name(0)
    except ImportError:
        try:
            cb_test = CatBoostClassifier(iterations=1, task_type="GPU", devices="0", verbose=0)
            cb_test.fit(np.array([[0, 1], [1, 0]]), np.array([0, 1]))
            has_gpu = True
            gpu_name = "CUDA GPU (CatBoost Verified)"
        except Exception:
            has_gpu = False

    print(f"  • GPU 가속 여부: {'🚀 사용 가능 (' + gpu_name + ')' if has_gpu else '💻 CPU 모드'}")
    return has_gpu, gpu_name

HAS_GPU, GPU_NAME = detect_gpu_environment()
print("=" * 70)


## 📂 2. 데이터 경로 자동 탐색 및 로드

1. 환경변수 `DATA_DIR`, `DATA_ZIP` 확인
2. `/kaggle/input` 내에서 `train.csv`, `test.csv`, `sample_submission.csv` 또는 `open.zip` 재귀 탐색
3. `open.zip` 발견 시 `BASE_WORKING_DIR / lg_data`로 자동 압축 해제하여 사용
4. Colab `/content/data`, 로컬 `data/` 등 다양한 경로 순차 탐색


In [ ]:
def find_and_prepare_data(base_working_dir):
    # 1. 환경변수 확인
    env_dir = os.environ.get("DATA_DIR")
    if env_dir and (Path(env_dir) / "train.csv").exists():
        return Path(env_dir)
    env_zip = os.environ.get("DATA_ZIP")
    if env_zip and Path(env_zip).exists():
        extract_to = base_working_dir / "lg_data"
        extract_to.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(env_zip, 'r') as zf:
            zf.extractall(extract_to)
        for r, _, f in os.walk(extract_to):
            if "train.csv" in f:
                return Path(r)

    # 2. Kaggle input 디렉토리 재귀 탐색
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for r, _, files in os.walk(kaggle_input):
            if "train.csv" in files and "test.csv" in files:
                print(f"🎯 Kaggle input 데이터 폴더 발견: {r}")
                return Path(r)
        for r, _, files in os.walk(kaggle_input):
            for f in files:
                if f.endswith(".zip") and ("open" in f.lower() or "data" in f.lower() or "aimers" in f.lower()):
                    zip_path = Path(r) / f
                    print(f"📦 Kaggle input zip 아카이브 발견: {zip_path} -> 압축 해제 중...")
                    extract_to = base_working_dir / "lg_data"
                    extract_to.mkdir(parents=True, exist_ok=True)
                    with zipfile.ZipFile(zip_path, 'r') as zf:
                        zf.extractall(extract_to)
                    for er, _, efiles in os.walk(extract_to):
                        if "train.csv" in efiles:
                            return Path(er)

    # 3. Colab / 로컬 데이터 디렉토리 후보군
    dir_candidates = [
        Path("data"),
        Path("../data"),
        Path(r"C:\Users\playj\Downloads\open\data"),
        Path("/content/data"),
        Path("/content/drive/MyDrive/data"),
        Path("/content/drive/MyDrive/open/data")
    ]
    for d in dir_candidates:
        if d.exists() and (d / "train.csv").exists():
            return d

    # 4. Colab / 로컬 zip 아카이브 후보군
    zip_candidates = [
        Path("open.zip"),
        Path("../open.zip"),
        Path(r"C:\Users\playj\Downloads\open.zip"),
        Path("/content/open.zip"),
        Path("/content/drive/MyDrive/open.zip")
    ]
    for z in zip_candidates:
        if z.exists():
            extract_to = base_working_dir / "lg_data"
            extract_to.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(extract_to)
            for er, _, efiles in os.walk(extract_to):
                if "train.csv" in efiles:
                    return Path(er)

    raise FileNotFoundError("데이터셋(train.csv 또는 open.zip)을 찾을 수 없습니다.")

RESOLVED_DATA_DIR = find_and_prepare_data(BASE_WORKING_DIR)
print(f"🎯 최종 확정 데이터 경로: {RESOLVED_DATA_DIR.resolve()}")

TARGET = "control_success"
ID = "row_id"
SHRINKAGE_K = 50

train = pd.read_csv(RESOLVED_DATA_DIR / "train.csv", encoding="utf-8-sig")
test = pd.read_csv(RESOLVED_DATA_DIR / "test.csv", encoding="utf-8-sig")
sample_sub = pd.read_csv(RESOLVED_DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

print(f"✅ train 로드 완료: {train.shape[0]:,}행 x {train.shape[1]}열 (시즌: {train['season'].min()}~{train['season'].max()})")
print(f"✅ test 로드 완료: {test.shape[0]:,}행 x {test.shape[1]}열")
print(f"✅ 전체 타깃 평균(제구 성공률): {train[TARGET].mean():.6f}")


## 🎯 3. 55개 고정 피처 정의 및 무결성 검증

- **규칙 준수**: 기존 `baseline` 55개 피처를 엄격하게 고정합니다 (추가/삭제/이름변경 불가).
- **Cold-Start 안전 보정**: `smooth_rate()`로 `NaN * 0 = NaN` 버그를 차단합니다.
- **Fold-Prior 격리**: 검증 세트 타깃 정보가 누수되지 않도록 학습 세트에서만 Prior를 산출합니다.


In [ ]:
# 1. 제외할 원본 컬럼 (7개)
REMOVAL_FEATURES = {
    "row_id",
    "asof_pitcher_prev5_game_success_rate",
    "asof_pitcher_pitchmix_n",
    "asof_pitcher_strike_rate",
    "asof_pitcher_fastball_rate",
    "run_total_before",
    "score_diff_home"
}

# 2. 기본 범주형 컬럼 (3개)
BASE_CAT_COLS = ["top_bottom", "game_type", "base_state"]

# 3. 14개 검증된 파생 피처 목록
BASE_DERIVED = [
    "pitcher_gap_prev1_career",
    "pitcher_gap_prev3_career",
    "pitcher_gap_prev5_career",
    "win_expectancy_dist50",
    "count_diff",
    "count_total",
    "same_hand_matchup",
    "pressure_x_recent_form",
    "runners_x_li",
    "batter_success_rate_shrunk",
    "reverse_rate_x_li",
    "middle_rate_x_count_diff",
    "late_inning_x_recent_form",
    "offspeed_x_li",
]

def estimate_feature_priors(frame):
    cols = [
        "asof_pitcher_success_rate",
        "asof_pitcher_reverse_rate",
        "asof_pitcher_middle_rate",
    ]
    priors = {c: float(frame[c].mean(skipna=True)) for c in cols if c in frame.columns}
    priors["target"] = float(frame[TARGET].mean())
    return priors

def smooth_rate(rate, n, prior, k=SHRINKAGE_K):
    n = pd.to_numeric(n, errors="coerce").fillna(0).clip(lower=0)
    rate = pd.to_numeric(rate, errors="coerce").fillna(prior)
    return (rate * n + prior * k) / (n + k)

def engineer_features(df, priors):
    df = df.copy()

    df["pitcher_gap_prev1_career"] = (
        df["asof_pitcher_prev1_game_success_rate"] - df["asof_pitcher_success_rate"]
    )
    df["pitcher_gap_prev3_career"] = (
        df["asof_pitcher_prev3_game_success_rate"] - df["asof_pitcher_success_rate"]
    )
    df["pitcher_gap_prev5_career"] = (
        df["asof_pitcher_prev5_game_success_rate"] - df["asof_pitcher_success_rate"]
    )
    df["win_expectancy_dist50"] = (df["home_win_expectancy"] - 50).abs()
    df["count_diff"] = df["balls_before"] - df["strikes_before"]
    df["count_total"] = df["balls_before"] + df["strikes_before"]
    df["same_hand_matchup"] = (df["pitcher_hand"] == df["batter_hand"]).astype("int8")
    df["pressure_x_recent_form"] = (
        df["count_diff"] * df["asof_pitcher_prev3_game_success_rate"]
    )
    df["runners_x_li"] = df["num_runners_on"] * df["li"]
    df["batter_success_rate_shrunk"] = smooth_rate(
        df["asof_batter_success_rate"],
        df["asof_batter_n"],
        priors["target"],
    )
    df["reverse_rate_x_li"] = df["asof_pitcher_reverse_rate"] * df["li"]
    df["middle_rate_x_count_diff"] = df["asof_pitcher_middle_rate"] * df["count_diff"]
    df["late_inning_x_recent_form"] = (
        (df["inning"] >= 7).astype("int8") * df["asof_pitcher_prev3_game_success_rate"]
    )
    df["offspeed_x_li"] = df["asof_pitcher_offspeed_rate"] * df["li"]

    return df

def get_55_feature_spec(test_columns):
    raw_kept = [c for c in test_columns if c not in REMOVAL_FEATURES]
    features = raw_kept + BASE_DERIVED
    cat_cols = [c for c in BASE_CAT_COLS if c in features]
    num_cols = [c for c in features if c not in cat_cols]
    
    assert len(features) == 55, f"피처 수가 55개가 아닙니다! 현재: {len(features)}개"
    assert len(features) == len(set(features)), "피처 목록에 중복된 이름이 존재합니다!"
    return features, cat_cols, num_cols

test_raw_cols = test.columns.tolist()
FEATURES_55, CAT_COLS_3, NUM_COLS_52 = get_55_feature_spec(test_raw_cols)

print("=" * 70)
print(f"✅ 피처 스펙 검증 완료: 총 {len(FEATURES_55)}개 피처")
print(f"  • 범주형 (3개): {CAT_COLS_3}")
print(f"  • 수치형 ({len(NUM_COLS_52)}개): {NUM_COLS_52[:8]} ... {NUM_COLS_52[-5:]}")
print("=" * 70)


## ⏳ 4. 검증 세트 분할 및 평가 지표 정의

- **검증 프로토콜**: 2019~2023년 정규 시즌(1,221,585행)으로 학습하고, 2024년 시즌(253,507행)을 단일 홀드아웃 검증 세트로 사용합니다.
- **평가 지표**:
  - Brier Score = mean((y_pred - y_true)**2)
  - Local BSS = max(0, 100000 * (1 - Brier / (r * (1 - r))))
  - 고정 편향 보정값(Calibration Shift): `CALIBRATION_SHIFT = -0.010462037831246366`


In [ ]:
# 2024 시즌 검증 홀드아웃 분할
is_val_2024 = train["season"].eq(2024)
raw_fit_2024 = train.loc[~is_val_2024].copy()
raw_val_2024 = train.loc[is_val_2024].copy()

fold_priors_2024 = estimate_feature_priors(raw_fit_2024)
train_featured_2024 = engineer_features(raw_fit_2024, fold_priors_2024)
val_featured_2024 = engineer_features(raw_val_2024, fold_priors_2024)

X_train_2024 = train_featured_2024[FEATURES_55]
y_train_2024 = raw_fit_2024[TARGET].to_numpy()

X_val_2024 = val_featured_2024[FEATURES_55]
y_val_2024 = raw_val_2024[TARGET].to_numpy()

val_target_rate = float(y_val_2024.mean())
baseline_brier_2024 = val_target_rate * (1.0 - val_target_rate)
FIXED_CALIBRATION_SHIFT = -0.010462037831246366

print(f"📊 검증 분할 완료:")
print(f"  • Train (2019~2023): {len(X_train_2024):,}행 | Prior: {fold_priors_2024['target']:.6f}")
print(f"  • Val (2024): {len(X_val_2024):,}행 | Target Rate: {val_target_rate:.6f}")
print(f"  • Baseline Brier r(1-r): {baseline_brier_2024:.6f}")

def compute_metrics(y_true, y_pred, model_name="Model", params=None, elapsed=0.0, device="CPU"):
    brier = float(np.mean((y_pred - y_true) ** 2))
    bss = max(0.0, 100000.0 * (1.0 - brier / baseline_brier_2024))
    
    shifted_pred = np.clip(y_pred + FIXED_CALIBRATION_SHIFT, 0.0, 1.0)
    shifted_brier = float(np.mean((shifted_pred - y_true) ** 2))
    shifted_bss = max(0.0, 100000.0 * (1.0 - shifted_brier / baseline_brier_2024))
    
    return {
        "model_name": model_name,
        "device": device,
        "n_features": len(FEATURES_55),
        "raw_brier": brier,
        "raw_bss": bss,
        "shifted_brier": shifted_brier,
        "shifted_bss": shifted_bss,
        "best_brier": min(brier, shifted_brier),
        "pred_mean": float(y_pred.mean()),
        "pred_min": float(y_pred.min()),
        "pred_max": float(y_pred.max()),
        "fit_seconds": elapsed,
        "params": str(params)
    }

def get_preprocessor(cat_cols, num_cols):
    return ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols),
    ])

def filter_params(p_dict):
    return {k: v for k, v in p_dict.items() if v is not None}


## 🔬 5. Phase 1: 55개 고정 피처 단일 모델 탐색 & 하이퍼파라미터 비교

- CatBoost는 GPU 가속 시 `task_type="GPU"`, `devices="0"`, `bootstrap_type="Bernoulli"`를 사용합니다.
- 점수표는 사전에 고정되지 않으며, 실제 셀 실행을 통해 동적으로 계산되어 누적됩니다.


In [ ]:
phase1_results = []
phase1_val_preds = {}

# 1. A. CatBoost 기준 모델 (Baseline 300 Trees)
print("\n[1/5] 🚀 CatBoost Baseline (300 Iterations) 학습 중...")
cb_base_params = filter_params({
    "random_seed": SEED,
    "objective": "Logloss",
    "verbose": 0,
    "learning_rate": 0.05,
    "n_estimators": 300,
    "max_depth": 6,
    "l2_leaf_reg": 3,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.7,
    "task_type": "GPU" if HAS_GPU else "CPU",
    "devices": "0" if HAS_GPU else None,
    "allow_writing_files": False,
})
cb_base_pipe = Pipeline([("pre", get_preprocessor(CAT_COLS_3, NUM_COLS_52)), ("clf", CatBoostClassifier(**cb_base_params))])

t0 = time.time()
cb_base_pipe.fit(X_train_2024, y_train_2024)
t_elapsed = time.time() - t0

pred_cb_base = cb_base_pipe.predict_proba(X_val_2024)[:, 1]
phase1_val_preds["CatBoost_Baseline_300"] = pred_cb_base
m_res = compute_metrics(y_val_2024, pred_cb_base, "CatBoost_Baseline_300", cb_base_params, t_elapsed, cb_base_params["task_type"])
phase1_results.append(m_res)
print(f"  ✅ Raw Brier: {m_res['raw_brier']:.6f} | Shifted: {m_res['shifted_brier']:.6f} | BSS: {m_res['raw_bss']:.2f} ({t_elapsed:.1f}s)")

# 2. B. 필수 CatBoost 후보 (500 Trees, Depth 8)
print("\n[2/5] 🚀 CatBoost Mandatory Candidate (500 Trees, Depth 8) 학습 중...")
cb_cand_params = filter_params({
    "random_seed": SEED,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "verbose": 0,
    "learning_rate": 0.03,
    "iterations": 500,
    "depth": 8,
    "l2_leaf_reg": 5,
    "random_strength": 1.0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.85,
    "task_type": "GPU" if HAS_GPU else "CPU",
    "devices": "0" if HAS_GPU else None,
    "allow_writing_files": False,
})
cb_cand_pipe = Pipeline([("pre", get_preprocessor(CAT_COLS_3, NUM_COLS_52)), ("clf", CatBoostClassifier(**cb_cand_params))])

t0 = time.time()
cb_cand_pipe.fit(X_train_2024, y_train_2024)
t_elapsed = time.time() - t0

pred_cb_cand = cb_cand_pipe.predict_proba(X_val_2024)[:, 1]
phase1_val_preds["CatBoost_500_Depth8"] = pred_cb_cand
m_res = compute_metrics(y_val_2024, pred_cb_cand, "CatBoost_500_Depth8", cb_cand_params, t_elapsed, cb_cand_params["task_type"])
phase1_results.append(m_res)
print(f"  ✅ Raw Brier: {m_res['raw_brier']:.6f} | Shifted: {m_res['shifted_brier']:.6f} | BSS: {m_res['raw_bss']:.2f} ({t_elapsed:.1f}s)")

# 3. C. CatBoost 추가 탐색 세트
print("\n[3/5] 🚀 CatBoost Targeted Hyperparameter Candidates 학습 중...")
cb_search_configs = [
    {"name": "CatBoost_Opt1_D7_LR04", "depth": 7, "iterations": 600, "learning_rate": 0.04, "l2_leaf_reg": 6, "subsample": 0.80, "random_strength": 0.8},
    {"name": "CatBoost_Opt2_D6_LR03_ES", "depth": 6, "iterations": 800, "learning_rate": 0.03, "l2_leaf_reg": 8, "subsample": 0.75, "random_strength": 1.2},
    {"name": "CatBoost_Opt3_D8_LR02_L210", "depth": 8, "iterations": 700, "learning_rate": 0.025, "l2_leaf_reg": 10, "subsample": 0.85, "random_strength": 1.5},
]

for cfg in cb_search_configs:
    c_name = cfg["name"]
    p_dict = filter_params({
        "random_seed": SEED,
        "loss_function": "Logloss",
        "verbose": 0,
        "iterations": cfg["iterations"],
        "depth": cfg["depth"],
        "learning_rate": cfg["learning_rate"],
        "l2_leaf_reg": cfg["l2_leaf_reg"],
        "bootstrap_type": "Bernoulli",
        "subsample": cfg["subsample"],
        "random_strength": cfg["random_strength"],
        "task_type": "GPU" if HAS_GPU else "CPU",
        "devices": "0" if HAS_GPU else None,
        "allow_writing_files": False,
    })
    pipe = Pipeline([("pre", get_preprocessor(CAT_COLS_3, NUM_COLS_52)), ("clf", CatBoostClassifier(**p_dict))])
    t0 = time.time()
    pipe.fit(X_train_2024, y_train_2024)
    t_elapsed = time.time() - t0
    
    pred = pipe.predict_proba(X_val_2024)[:, 1]
    phase1_val_preds[c_name] = pred
    m_res = compute_metrics(y_val_2024, pred, c_name, p_dict, t_elapsed, p_dict["task_type"])
    phase1_results.append(m_res)
    print(f"  • {c_name} => Raw Brier: {m_res['raw_brier']:.6f} | Shifted: {m_res['shifted_brier']:.6f} ({t_elapsed:.1f}s)")

# 4. D. LightGBM Candidates
print("\n[4/5] 🚀 LightGBM Candidates 학습 중...")
lgb_configs = [
    {"name": "LightGBM_Base_233", "n_estimators": 233, "learning_rate": 0.03, "num_leaves": 31, "max_depth": 6, "min_child_samples": 300, "reg_alpha": 1.0, "reg_lambda": 2.0, "subsample": 0.7, "colsample_bytree": 0.6},
    {"name": "LightGBM_D7_Leaves63", "n_estimators": 400, "learning_rate": 0.03, "num_leaves": 63, "max_depth": 7, "min_child_samples": 200, "reg_alpha": 2.0, "reg_lambda": 5.0, "subsample": 0.8, "colsample_bytree": 0.7},
]

for cfg in lgb_configs:
    l_name = cfg["name"]
    p_dict = {
        "random_state": SEED,
        "objective": "binary",
        "n_jobs": -1,
        "verbosity": -1,
        "n_estimators": cfg["n_estimators"],
        "learning_rate": cfg["learning_rate"],
        "num_leaves": cfg["num_leaves"],
        "max_depth": cfg["max_depth"],
        "min_child_samples": cfg["min_child_samples"],
        "reg_alpha": cfg["reg_alpha"],
        "reg_lambda": cfg["reg_lambda"],
        "subsample": cfg["subsample"],
        "colsample_bytree": cfg["colsample_bytree"],
    }
    pipe = Pipeline([("pre", get_preprocessor(CAT_COLS_3, NUM_COLS_52)), ("clf", LGBMClassifier(**p_dict))])
    t0 = time.time()
    pipe.fit(X_train_2024, y_train_2024)
    t_elapsed = time.time() - t0
    
    pred = pipe.predict_proba(X_val_2024)[:, 1]
    phase1_val_preds[l_name] = pred
    m_res = compute_metrics(y_val_2024, pred, l_name, p_dict, t_elapsed, "CPU")
    phase1_results.append(m_res)
    print(f"  • {l_name} => Raw Brier: {m_res['raw_brier']:.6f} | Shifted: {m_res['shifted_brier']:.6f} ({t_elapsed:.1f}s)")

# 5. E. XGBoost Candidate
print("\n[5/5] 🚀 XGBoost Candidate 학습 중...")
x_name = "XGBoost_Hist_D6"
p_dict_xgb = {
    "random_state": SEED,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_jobs": -1,
    "tree_method": "hist",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_lambda": 2.0,
}
pipe_xgb = Pipeline([("pre", get_preprocessor(CAT_COLS_3, NUM_COLS_52)), ("clf", XGBClassifier(**p_dict_xgb))])
t0 = time.time()
pipe_xgb.fit(X_train_2024, y_train_2024)
t_elapsed = time.time() - t0

pred_xgb = pipe_xgb.predict_proba(X_val_2024)[:, 1]
phase1_val_preds[x_name] = pred_xgb
m_res = compute_metrics(y_val_2024, pred_xgb, x_name, p_dict_xgb, t_elapsed, "CPU")
phase1_results.append(m_res)
print(f"  • {x_name} => Raw Brier: {m_res['raw_brier']:.6f} | Shifted: {m_res['shifted_brier']:.6f} ({t_elapsed:.1f}s)")

# Phase 1 Summary
phase1_df = pd.DataFrame(phase1_results)
baseline_ref_brier = phase1_df.loc[phase1_df["model_name"] == "CatBoost_Baseline_300", "best_brier"].values[0]
phase1_df["brier_gain_vs_baseline"] = baseline_ref_brier - phase1_df["best_brier"]
phase1_df = phase1_df.sort_values("best_brier").reset_index(drop=True)

print("\n" + "=" * 80)
print("📊 Phase 1: 55개 고정 피처 2024 검증 모델 비교 결과 (Brier 순위)")
print("=" * 80)
display(phase1_df[["model_name", "device", "raw_brier", "shifted_brier", "best_brier", "brier_gain_vs_baseline", "raw_bss", "fit_seconds"]])


## ⏳ 6. Phase 2: 다중 시즌 시계열 확장 검증 (Forward Validation)

2024 단일 연도 우연에 의한 과적합을 방지하기 위해, Phase 1 상위 2~3개 아키텍처에 대해 **3개 시즌 전진 검증 (Forward Validation)**을 수행합니다.

- **Fold 1 (2022 검증)**: Train 2019~2021 (728,790행) -> Val 2022 (247,443행)
- **Fold 2 (2023 검증)**: Train 2019~2022 (976,233행) -> Val 2023 (245,352행)
- **Fold 3 (2024 검증)**: Train 2019~2023 (1,221,585행) -> Val 2024 (253,507행)


In [ ]:
FORWARD_FOLDS = [
    ("Val_2022", [2019, 2020, 2021], 2022),
    ("Val_2023", [2019, 2020, 2021, 2022], 2023),
    ("Val_2024", [2019, 2020, 2021, 2022, 2023], 2024),
]

top_candidates = [
    ("Best_CatBoost", CatBoostClassifier(**cb_cand_params)),
    ("Baseline_CatBoost", CatBoostClassifier(**cb_base_params)),
    ("Best_LightGBM", LGBMClassifier(**lgb_configs[1])),
]

forward_records = []
forward_predictions = {name: {} for name, _ in top_candidates}

for fold_name, train_years, val_year in FORWARD_FOLDS:
    print(f"\n--- 🚀 Running {fold_name} (Train: {train_years}, Val: {val_year}) ---")
    tr_mask = train["season"].isin(train_years)
    va_mask = train["season"].eq(val_year)
    
    r_fit = train.loc[tr_mask].copy()
    r_val = train.loc[va_mask].copy()
    
    priors_fold = estimate_feature_priors(r_fit)
    f_tr = engineer_features(r_fit, priors_fold)[FEATURES_55]
    f_va = engineer_features(r_val, priors_fold)[FEATURES_55]
    
    y_tr = r_fit[TARGET].to_numpy()
    y_va = r_val[TARGET].to_numpy()
    v_rate = float(y_va.mean())
    base_brier = v_rate * (1.0 - v_rate)
    
    for cand_name, clf_obj in top_candidates:
        pipe = Pipeline([("pre", get_preprocessor(CAT_COLS_3, NUM_COLS_52)), ("clf", sklearn.base.clone(clf_obj))])
        t0 = time.time()
        pipe.fit(f_tr, y_tr)
        t_el = time.time() - t0
        
        pred = pipe.predict_proba(f_va)[:, 1]
        forward_predictions[cand_name][val_year] = (y_va, pred)
        
        brier = float(np.mean((pred - y_va) ** 2))
        bss = max(0.0, 100000.0 * (1.0 - brier / base_brier))
        
        forward_records.append({
            "fold": fold_name,
            "val_season": val_year,
            "candidate": cand_name,
            "brier": brier,
            "bss": bss,
            "val_target_rate": v_rate,
            "pred_mean": float(pred.mean()),
            "fit_seconds": t_el
        })
        print(f"  [{cand_name}] Brier: {brier:.6f} | BSS: {bss:.2f} ({t_el:.1f}s)")

forward_df = pd.DataFrame(forward_records)
forward_summary = forward_df.groupby("candidate").agg(
    Mean_Brier=("brier", "mean"),
    Brier_2024=("brier", lambda x: x.iloc[-1]),
    Mean_BSS=("bss", "mean"),
    Total_Fit_Time=("fit_seconds", "sum")
).reset_index().sort_values("Mean_Brier")

print("\n" + "=" * 70)
print("📊 다중 시즌 (2022~2024) Forward Validation 종합 요약")
print("=" * 70)
display(forward_summary)


## 🔀 7. Phase 3: 앙상블 (Probability Blending) 탐색

Top CatBoost와 Top LightGBM의 예측 확률 가중 평균을 Forward Validation 상에서 탐색합니다.
- 가중치 w in [0.0, 0.1, 0.2, ..., 1.0] 그리드 평가
- 단일 최고 모델 대비 최소 0.00001 이상 Brier가 개선될 때만 앙상블을 최종 채택합니다.


In [ ]:
weights = np.linspace(0.0, 1.0, 11)
blend_records = []

for w in weights:
    fold_briers = []
    for val_year in [2022, 2023, 2024]:
        y_va, pred_cb = forward_predictions["Best_CatBoost"][val_year]
        _, pred_lgb = forward_predictions["Best_LightGBM"][val_year]
        
        pred_blend = w * pred_cb + (1.0 - w) * pred_lgb
        brier_blend = float(np.mean((pred_blend - y_va) ** 2))
        fold_briers.append(brier_blend)
    
    mean_blend_brier = float(np.mean(fold_briers))
    blend_records.append({
        "weight_catboost": round(w, 2),
        "weight_lightgbm": round(1.0 - w, 2),
        "mean_forward_brier": mean_blend_brier,
        "brier_2024": fold_briers[-1]
    })

blend_df = pd.DataFrame(blend_records).sort_values("mean_forward_brier").reset_index(drop=True)
best_single_brier = forward_summary["Mean_Brier"].min()
blend_df["gain_vs_best_single"] = best_single_brier - blend_df["mean_forward_brier"]

print("=" * 70)
print("📊 앙상블 블렌딩 가중치 탐색 결과 (상위 5개 조합)")
print("=" * 70)
display(blend_df.head(5))

best_blend_gain = blend_df.loc[0, "gain_vs_best_single"]
if best_blend_gain >= 0.00001:
    print(f"🎉 앙상블 채택: 가중치 CatBoost {blend_df.loc[0, 'weight_catboost']} + LightGBM {blend_df.loc[0, 'weight_lightgbm']} (Brier 개선량: {best_blend_gain:.6f})")
    FINAL_MODEL_TYPE = "Ensemble_CatBoost_LightGBM"
    FINAL_ENSEMBLE_WEIGHTS = (float(blend_df.loc[0, "weight_catboost"]), float(blend_df.loc[0, "weight_lightgbm"]))
else:
    print(f"ℹ️ 단일 모델 채택: 앙상블 개선폭({best_blend_gain:.6f})이 유의 기준(0.00001) 미만이므로 더 단순하고 안전한 최고 단일 모델을 채택합니다.")
    FINAL_MODEL_TYPE = "Single_CatBoost"
    FINAL_ENSEMBLE_WEIGHTS = (1.0, 0.0)


## 🏆 8. Phase 4: 전체 데이터(2019~2024) 최종 재학습 & 네이티브 모델 저장

- 전체 1,475,092행 데이터로 Prior를 산출하고 최종 재학습을 진행합니다.
- 평가 서버 호환성을 위해 **CatBoost 네이티브(`.cbm`) / LightGBM 네이티브(`.txt`) 모델 파일 및 `metadata.json`**으로 분리 내보냅니다.


In [ ]:
if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1. 전체 데이터(2019~2024) Prior 산출 및 Feature Engineering
full_priors = estimate_feature_priors(train)
full_train_df = engineer_features(train, full_priors)

X_full = full_train_df[FEATURES_55]
y_full = train[TARGET].to_numpy()

print(f"🚀 전체 {len(X_full):,}행 데이터로 최종 모델 재학습 시작...")

# 2. 전처리 파이프라인 학습
final_preprocessor = get_preprocessor(CAT_COLS_3, NUM_COLS_52)
final_preprocessor.fit(X_full)

encoder = final_preprocessor.named_transformers_["cat"]
imputer = final_preprocessor.named_transformers_["num"]

cat_categories_list = [list(map(str, cats.tolist())) for cats in encoder.categories_]
num_medians_list = [float(x) for x in imputer.statistics_]

# 3. 모델 학습 및 네이티브 저장
X_full_transformed = final_preprocessor.transform(X_full)

if FINAL_MODEL_TYPE == "Single_CatBoost":
    final_cb = CatBoostClassifier(**cb_cand_params)
    t0 = time.time()
    final_cb.fit(X_full_transformed, y_full)
    print(f"  ✅ CatBoost 전체 재학습 완료 ({time.time() - t0:.1f}초)")
    final_cb.save_model(MODEL_DIR / "model_cb.cbm")
    req_content = "catboost==1.2.10\n"
    
elif FINAL_MODEL_TYPE == "Ensemble_CatBoost_LightGBM":
    final_cb = CatBoostClassifier(**cb_cand_params)
    t0 = time.time()
    final_cb.fit(X_full_transformed, y_full)
    final_cb.save_model(MODEL_DIR / "model_cb.cbm")
    
    final_lgb = LGBMClassifier(**lgb_configs[1])
    final_lgb.fit(X_full_transformed, y_full)
    final_lgb.booster_.save_model(str(MODEL_DIR / "model_lgb.txt"))
    print(f"  ✅ 앙상블 (CatBoost + LightGBM) 재학습 및 네이티브 저장 완료 ({time.time() - t0:.1f}초)")
    req_content = "catboost==1.2.10\nlightgbm==4.7.0\n"

# 4. metadata.json 저장
metadata = {
    "model_type": FINAL_MODEL_TYPE,
    "ensemble_weights": FINAL_ENSEMBLE_WEIGHTS,
    "features": FEATURES_55,
    "cat_cols": CAT_COLS_3,
    "num_cols": NUM_COLS_52,
    "cat_categories": cat_categories_list,
    "num_medians": num_medians_list,
    "priors": full_priors,
    "calibration_shift": FIXED_CALIBRATION_SHIFT,
    "shrinkage_k": SHRINKAGE_K,
    "package_versions": {
        "catboost": catboost.__version__,
        "lightgbm": lgb.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    }
}

with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

(BUILD_DIR / "requirements.txt").write_text(req_content, encoding="utf-8")
print(f"✅ 모델 및 메타데이터 저장 완료: {MODEL_DIR}")


## 📦 9. Phase 5: 안전 제출 `script.py` 자동 생성 및 DACON `submit.zip` 패키징

- Single-Source-of-Truth 방식으로 `engineer_features()` 함수 원본을 `script.py` 내부에 직접 포함합니다.
- `sample_submission.csv`의 `row_id` 순서와 1:1 완벽 정합성을 보장합니다.
- 최종 선택이 앙상블이면 `model_cb.cbm`과 `model_lgb.txt`를 모두 로드하여 가중 블렌딩하고, 단일 모델이면 `model_cb.cbm`을 단독 로드합니다.


In [ ]:
engineer_src = inspect.getsource(engineer_features)
smooth_src = inspect.getsource(smooth_rate)

SCRIPT_CONTENT = f'''# script.py — LG Aimers DACON 55-Feature Native Inference Script
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parent
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "model"
OUTPUT_DIR = ROOT / "output"
ID = "row_id"
TARGET = "control_success"
SHRINKAGE_K = {SHRINKAGE_K!r}

{smooth_src}

{engineer_src}

def transform_for_model(frame, metadata):
    parts = []
    for col, categories in zip(metadata["cat_cols"], metadata["cat_categories"]):
        mapping = {{value: idx for idx, value in enumerate(categories)}}
        encoded = frame[col].astype(str).map(mapping).fillna(-1).to_numpy(dtype=np.float64)
        parts.append(encoded.reshape(-1, 1))

    numeric = frame[metadata["num_cols"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    medians = np.asarray(metadata["num_medians"], dtype=np.float64)
    missing_rows, missing_cols = np.where(np.isnan(numeric))
    numeric[missing_rows, missing_cols] = medians[missing_cols]
    parts.append(numeric)
    return np.hstack(parts)


def main():
    with open(MODEL_DIR / "metadata.json", encoding="utf-8") as f:
        metadata = json.load(f)

    test = pd.read_csv(DATA_DIR / "test.csv", encoding="utf-8-sig")
    sample = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
    if not test[ID].is_unique or not sample[ID].is_unique:
        raise ValueError("row_id 중복을 발견했습니다.")

    featured = engineer_features(test, metadata["priors"])
    missing_features = sorted(set(metadata["features"]) - set(featured.columns))
    if missing_features:
        raise ValueError(f"입력 피처 누락: {{missing_features}}")
    
    matrix = transform_for_model(featured[metadata["features"]], metadata)

    model_type = metadata.get("model_type", "Single_CatBoost")
    if model_type == "Single_CatBoost":
        cb_model = CatBoostClassifier()
        cb_model.load_model(MODEL_DIR / "model_cb.cbm")
        predictions = cb_model.predict_proba(matrix)[:, 1]
    elif model_type == "Ensemble_CatBoost_LightGBM":
        import lightgbm as lgb
        w_cb, w_lgb = metadata["ensemble_weights"]
        cb_model = CatBoostClassifier()
        cb_model.load_model(MODEL_DIR / "model_cb.cbm")
        pred_cb = cb_model.predict_proba(matrix)[:, 1]
        
        lgb_booster = lgb.Booster(model_file=str(MODEL_DIR / "model_lgb.txt"))
        pred_lgb = lgb_booster.predict(matrix)
        predictions = w_cb * pred_cb + w_lgb * pred_lgb
    else:
        raise ValueError(f"알 수 없는 model_type: {{model_type}}")

    # Calibration Shift 적용 및 [0, 1] Clipping
    shift_val = float(metadata.get("calibration_shift", 0.0))
    predictions = np.clip(predictions + shift_val, 0.0, 1.0)

    # sample_submission.csv의 row_id 순서에 맞추어 1:1 안전 매핑
    pred_by_id = pd.Series(predictions, index=test[ID], name=TARGET)
    submission = sample[[ID]].copy()
    submission[TARGET] = submission[ID].map(pred_by_id)

    # 무결성 검증
    if len(submission) != len(test):
        raise ValueError(f"행 수 불일치: submission={{len(submission)}}, test={{len(test)}}")
    if submission[TARGET].isna().any():
        raise ValueError("예측값에 결측치(NaN)가 존재합니다.")
    if not np.isfinite(submission[TARGET].to_numpy()).all():
        raise ValueError("예측값에 무한대(inf)가 존재합니다.")
    if not submission[TARGET].between(0, 1).all():
        raise ValueError("예측 확률이 [0, 1] 범위를 벗어났습니다.")
    if list(submission.columns) != [ID, TARGET]:
        raise ValueError(f"제출 컬럼 오류: {{submission.columns.tolist()}}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_DIR / "submission.csv", index=False, encoding="utf-8-sig")
    print(f"[Done] submission.csv created: {{len(submission):,}} rows")


if __name__ == "__main__":
    main()
'''

(BUILD_DIR / "script.py").write_text(SCRIPT_CONTENT, encoding="utf-8")
compile(SCRIPT_CONTENT, "script.py", "exec")
print("✅ script.py 생성 및 Python 문법 검사 통과!")

# ZIP 아카이브 압축 생성
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(BUILD_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(BUILD_DIR).as_posix())

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()
    top = {name.split("/")[0] for name in names}
    assert top == {"model", "script.py", "requirements.txt"}, f"잘못된 ZIP 구조: {top}"
    assert "data" not in top and "output" not in top, "금지된 디렉토리가 포함되었습니다!"
    print(f"✅ ZIP 구조 검증 완료: {names}")
    print(f"📦 생성된 최종 제출 파일: {ZIP_PATH} (크기: {ZIP_PATH.stat().st_size / 1024:.1f} KB)")


## 🔍 10. Phase 6: 독립 격리 환경 사전 추론 검증 (Smoke Test)

생성된 `submit_v02.zip`을 완전히 독립된 임시 폴더에 해제하고, 별도의 서브프로세스에서 `python script.py`를 실행하여 실제 채점 서버와 동일한 조건에서 추론을 검증합니다.


In [ ]:
import subprocess

SMOKE_DIR = ARTIFACT_DIR / "smoke_test"
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(SMOKE_DIR)

(SMOKE_DIR / "data").mkdir(parents=True, exist_ok=True)
(SMOKE_DIR / "output").mkdir(parents=True, exist_ok=True)

test.to_csv(SMOKE_DIR / "data" / "test.csv", index=False, encoding="utf-8-sig")
sample_sub.to_csv(SMOKE_DIR / "data" / "sample_submission.csv", index=False, encoding="utf-8-sig")

print(f"🚀 격리 폴더({SMOKE_DIR})에서 script.py 단독 실행 검증 시작...")
t0 = time.time()
res = subprocess.run(
    [sys.executable, str(SMOKE_DIR / "script.py")],
    cwd=str(SMOKE_DIR),
    capture_output=True,
    text=True,
    timeout=300
)
t_elapsed = time.time() - t0

print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f"❌ 독립 추론 실패 (Exit Code: {res.returncode})")

# 생성된 submission.csv 확인
gen_sub = pd.read_csv(SMOKE_DIR / "output" / "submission.csv")
assert len(gen_sub) == len(sample_sub), "행 수 불일치"
assert gen_sub[ID].tolist() == sample_sub[ID].tolist(), "row_id 순서 불일치"
assert gen_sub[TARGET].notna().all() and gen_sub[TARGET].between(0, 1).all(), "예측값 이상"
print(f"✅ 독립 추론 검증 완벽 통과! (소요 시간: {t_elapsed:.2f}초)")

# SHA256 체크섬 계산
sha256_hash = hashlib.sha256()
with open(ZIP_PATH, "rb") as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)
zip_sha256 = sha256_hash.hexdigest()
print(f"🔒 최종 제출 ZIP SHA256: {zip_sha256}")


## 📊 11. 최종 보고서 및 설정 파일 자동 내보내기

실제 실행 결과에 기반하여 다음 산출물을 동적으로 기록합니다.
- `reports/model_tuning_results.csv`
- `reports/model_selection_summary.md`
- `artifacts/final_config.json`
- `artifacts/package_validation.json`


In [ ]:
# 1. 튜닝 결과 테이블 CSV 저장
phase1_df.to_csv(REPORTS_DIR / "model_tuning_results.csv", index=False, encoding="utf-8-sig")

# 2. final_config.json 저장
final_config_data = {
    "selected_model_type": FINAL_MODEL_TYPE,
    "feature_count": len(FEATURES_55),
    "features": FEATURES_55,
    "calibration_shift": FIXED_CALIBRATION_SHIFT,
    "catboost_best_params": cb_cand_params,
    "lightgbm_best_params": lgb_configs[1],
    "ensemble_weights": FINAL_ENSEMBLE_WEIGHTS,
    "brier_gain_vs_baseline": float(phase1_df.loc[0, "brier_gain_vs_baseline"]),
    "zip_path": str(ZIP_PATH),
    "zip_sha256": zip_sha256
}
with open(ARTIFACT_DIR / "final_config.json", "w", encoding="utf-8") as f:
    json.dump(final_config_data, f, ensure_ascii=False, indent=2)

# 3. package_validation.json 저장
val_report = {
    "zip_path": str(ZIP_PATH),
    "zip_size_bytes": ZIP_PATH.stat().st_size,
    "zip_sha256": zip_sha256,
    "top_level_structure": ["model/", "script.py", "requirements.txt"],
    "smoke_test_passed": True,
    "features_verified_count": 55,
    "submission_rows": len(sample_sub),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
}
with open(ARTIFACT_DIR / "package_validation.json", "w", encoding="utf-8") as f:
    json.dump(val_report, f, ensure_ascii=False, indent=2)

# 4. model_selection_summary.md 작성
summary_md = f'''# ⚾ LG Aimers 모델 튜닝 및 최종 선택 요약 보고서

## 1. 개요 및 실험 원칙
- **피처 스펙**: 검증된 `baseline` 55개 피처 완전 고정 (3개 범주형, 52개 수치형)
- **검증 방식**: 2024 시즌 홀드아웃(Phase 1) + 2022~2024 3개 시즌 Forward Validation(Phase 2)
- **최종 선택 모델**: `{FINAL_MODEL_TYPE}`
- **최종 패키지 파일**: `{ZIP_PATH}` (SHA256: `{zip_sha256}`)

## 2. Phase 1: 55개 고정 피처 단일 모델 비교
- **기준 CatBoost (300 iter)**: Brier `{phase1_df.loc[phase1_df['model_name']=='CatBoost_Baseline_300', 'best_brier'].values[0]:.6f}`
- **필수 CatBoost (500 iter, Depth 8)**: Brier `{phase1_df.loc[phase1_df['model_name']=='CatBoost_500_Depth8', 'best_brier'].values[0]:.6f}`
- **최고 단일 모델**: `{phase1_df.loc[0, 'model_name']}` (Brier `{phase1_df.loc[0, 'best_brier']:.6f}`, 기준 대비 개선량 `{phase1_df.loc[0, 'brier_gain_vs_baseline']:.6f}`)

## 3. 다중 시즌 Forward Validation & 앙상블 결과
- **2022~2024 Forward Validation Mean Brier**: `{forward_summary.loc[0, 'Mean_Brier']:.6f}`
- **최종 선택 모델**: `{FINAL_MODEL_TYPE}` (가중치: {FINAL_ENSEMBLE_WEIGHTS})

## 4. 제출 패키지 무결성 검증
- **구조**: `model/` (네이티브 모델 및 `metadata.json`), `script.py`, `requirements.txt`
- **독립 격리 추론 검증**: 완료 (1:1 `sample_submission.csv` 매핑 및 결측/이상치 없음 확인)
'''
(REPORTS_DIR / "model_selection_summary.md").write_text(summary_md, encoding="utf-8")

print("=" * 70)
print("🎉 모든 산출물 생성 및 검증 완료!")
print(f"  • Notebook: LG_Aimers_model_tuning_55features.ipynb")
print(f"  • Reports: {REPORTS_DIR / 'model_tuning_results.csv'}, {REPORTS_DIR / 'model_selection_summary.md'}")
print(f"  • Artifacts: {ARTIFACT_DIR / 'final_config.json'}, {ARTIFACT_DIR / 'package_validation.json'}, {ZIP_PATH}")
print("=" * 70)
